In [ ]:
# import and setup a model
import torch
from ultralytics import YOLO
import numpy as np

yolo = YOLO("yolov5nu.pt")
net = yolo.model

In [ ]:
# # Here are all functions used in the runtime - they appered as they in the __init__
net

In [ ]:
# sniff a runtime
execution = []
no_dup_execution = []

# to remove duplicates
seen = set()

def add_hook(module, name):
    def hook(module, input, output):
        execution.append(name)
    return hook

for name, m in net.named_modules():
    if isinstance(m, (torch.nn.Conv2d, torch.nn.BatchNorm2d, torch.nn.SiLU,
                      torch.nn.MaxPool2d, torch.nn.Upsample)):
        m.register_forward_hook(add_hook(m, name))

# a forward pass
x = torch.randn(1, 3, 640, 640)
net(x)

for layer in execution:
    if layer not in seen:
        no_dup_execution.append(layer)
        seen.add(layer)



print(execution)
duplicates = [x for x in execution if execution.count(x) > 1]
print(any(execution.count(x) > 1 for x in execution))
print(duplicates)
print(no_dup_execution)
len(no_dup_execution)


## We cant really build map using this as python may reuse functions and some layers are not defined using function

In [ ]:
# layer_nums_back_c3 = use net
layer_nums_c3 = []
layer_nums_sppf = []

for i, layer in enumerate(net.model):
    name = layer.__class__.__name__.lower()
    if 'c3' in name:
        layer_nums_c3.append(i)
    if 'sppf' in name:
        layer_nums_sppf.append(i)

layer_nums_back_c3 = layer_nums_c3[:4]

print("C3 layers:", layer_nums_c3)
print("C3 back layers:", layer_nums_back_c3)
print("SPPF layers:", layer_nums_sppf)

In [ ]:
def flatten_atomic(module, prefix=""):
    atomic_layers = []
    for name, sub in module.named_children():
        full_name = f"{prefix}.{name}" if prefix else name
        if list(sub.children()):
            atomic_layers.extend(flatten_atomic(sub, prefix=full_name))
        else:
            atomic_layers.append({"full_name": full_name, "model": sub})
    return atomic_layers

atomic_layers_unordered_incomplete = flatten_atomic(net.model)
atomic_layers_unordered_incomplete

In [ ]:
# reorder C3 block as we heard in sniffing
# policy: cv1 -> cv2 -> cv3 -> m to cv1 -> m -> cv2 -> cv3
def reorder_c3_blocks(layers):
    reordered = []
    c3_buffer = []
    for layer in layers:
        name = layer['full_name']
        ln = int(name.split('.')[0])
        if any(x in name for x in ['.cv1.', '.cv2.', '.cv3.', '.m.']) and ln in layer_nums_c3:
            c3_buffer.append(layer)
        else:
            if c3_buffer:
                cv1 = [l for l in c3_buffer if '.cv1.' in l['full_name'] and '.m.' not in l['full_name']]
                cv2 = [l for l in c3_buffer if '.cv2.' in l['full_name'] and '.m.' not in l['full_name']]
                cv3 = [l for l in c3_buffer if '.cv3.' in l['full_name'] and '.m.' not in l['full_name']]
                m   = [l for l in c3_buffer if '.m.' in l['full_name']]
                reordered.extend(cv1 + m + cv2 + cv3)
                c3_buffer = []
            reordered.append(layer)
    if c3_buffer:
        cv1 = [l for l in c3_buffer if '.cv1.' in l['full_name'] and '.m.' not in l['full_name']]
        cv2 = [l for l in c3_buffer if '.cv2.' in l['full_name'] and '.m.' not in l['full_name']]
        cv3 = [l for l in c3_buffer if '.cv3.' in l['full_name'] and '.m.' not in l['full_name']]
        m   = [l for l in c3_buffer if '.m.' in l['full_name']]
        reordered.extend(cv1 + m + cv2 + cv3)
    return reordered

atomic_layer_ordered = reorder_c3_blocks(atomic_layers_unordered_incomplete)
atomic_layer_ordered


# reorder SPPF block as we heard in sniffing
# policy: cv1 -> cv2 -> m to cv1 -> m -> cv2
def reorder_sppf_blocks(layers):
    reordered = []
    sppf_buffer = []
    for layer in layers:
        name = layer['full_name']
        ln = int(name.split('.')[0])
        if any(x in name for x in ['.cv1', '.cv2', '.m']) and ln in layer_nums_sppf:
            sppf_buffer.append(layer)
        else:
            if sppf_buffer:
                cv1 = [l for l in sppf_buffer if '.cv1' in l['full_name'] and '.m' not in l['full_name']]
                cv2 = [l for l in sppf_buffer if '.cv2' in l['full_name'] and '.m' not in l['full_name']]
                m   = [l for l in sppf_buffer if '.m' in l['full_name']]
                reordered.extend(cv1 + m + cv2)
                sppf_buffer = []
            reordered.append(layer)
    if sppf_buffer:
        cv1 = [l for l in sppf_buffer if '.cv1' in l['full_name'] and '.m' not in l['full_name']]
        cv2 = [l for l in sppf_buffer if '.cv.' in l['full_name'] and '.m' not in l['full_name']]
        m   = [l for l in sppf_buffer if '.m.' in l['full_name']]
        reordered.extend(cv1 + m + cv2)
    return reordered

atomic_layer_ordered = reorder_sppf_blocks(atomic_layer_ordered)
atomic_layer_ordered

In [ ]:
def debug_compare(list1, list2):
    if len(list1) != len(list2):
        print(f"Warning: Lists have different lengths: {len(list1)} vs {len(list2)}")
    for i, (a, b) in enumerate(zip(list1, list2)):
        if a == b:
            print(f"\nIndex {i}: \nmatches: {a}")
        else:
            print(f"\nIndex {i}:\nelem1: {a}, \nelem2: {b}")

debug_compare(atomic_layers_unordered_incomplete, atomic_layer_ordered)

In [ ]:
# add missing concat, res_add
# policy:
# add a {'full_name': 'x', 'model': Concat()}, // literary an x letter 
# 1. just before cv3.conv in c3 blocks
# 2. just before cv2.conv in sppf blocks
# add a {'full_name': 'x', 'model': "ResAdd()"}, // literary an x letter
# just before previously add Concat if in c3_back


from copy import deepcopy
from torch import nn

atomic_layers_ordered_completed = deepcopy(atomic_layer_ordered)
i = 0
while i < len(atomic_layers_ordered_completed):
    layer = atomic_layers_ordered_completed[i]
    name = layer['full_name'].lower()
    ln = int(name.split('.')[0])
    is_c3_cv3 = ('.cv3.conv' in name and ln in layer_nums_c3)
    is_sppf_cv2 = ('.cv2.conv' in name and ln in layer_nums_sppf)
    if (is_c3_cv3 or is_sppf_cv2) and ('.m' not in name):
        if is_c3_cv3 and ln in layer_nums_back_c3:
            atomic_layers_ordered_completed.insert(i, {'full_name': 'x', 'model': "resadd"})
            i += 1
        atomic_layers_ordered_completed.insert(i, {'full_name': 'x', 'model': "concat"})
        i += 1
    i += 1

atomic_layers_ordered_completed

In [ ]:
debug_compare(atomic_layers_unordered_incomplete, atomic_layers_ordered_completed)

In [ ]:
atomic_layers = atomic_layers_unordered_incomplete
atomic_layers_ordered_completed

In [ ]:
from typing import List, Dict

def generate_config(layers: List[Dict]) -> List[List[int]]:
    config=[]
    for layer in layers:
        inner=[]
        model=layer['model']
        op_type='unknown'
        if isinstance(model,str):
            op_type=model.lower()
        else:
            cls_name=model.__class__.__name__.lower()
            if 'conv' in cls_name:op_type='conv'
            elif 'batchnorm' in cls_name:op_type='bn'
            elif 'relu' in cls_name or 'silu' in cls_name:op_type='activation'
            elif 'maxpool' in cls_name:op_type='pool'
            elif 'upsample' in cls_name:op_type='upsample'
            elif 'concat' in cls_name:op_type='concat'
            elif 'resadd' in cls_name:op_type='resadd'
            elif 'c3' in cls_name:op_type='c3'
            elif 'sppf' in cls_name:op_type='sppf'
            elif 'detect' in cls_name:op_type='detect'
        inner.append(op_type)
        if op_type=='conv':
            in_ch=model.in_channels
            out_ch=model.out_channels
            k_h,k_w=model.kernel_size if isinstance(model.kernel_size,tuple) else (model.kernel_size,model.kernel_size)
            s_h,s_w=model.stride if isinstance(model.stride,tuple) else (model.stride,model.stride)
            p_h,p_w=model.padding if isinstance(model.padding,tuple) else (model.padding,model.padding)
            bias=model.bias is not None
            inner.extend([in_ch,out_ch,k_h,k_w,s_h,s_w,p_h,p_w,bias])
        elif op_type=='bn':
            num_feat=model.num_features
            eps=model.eps
            momentum=model.momentum
            affine=model.affine
            track_stats=model.track_running_stats
            inner.extend([num_feat,eps,momentum,affine,track_stats])
        elif op_type=='activation':
            inplace=getattr(model,'inplace',False)
            inner.append(inplace)
        elif op_type=='pool':
            k_h,k_w=model.kernel_size if isinstance(model.kernel_size,tuple) else (model.kernel_size,model.kernel_size)
            s_h,s_w=model.stride if isinstance(model.stride,tuple) else (model.stride,model.stride)
            p_h,p_w=model.padding if isinstance(model.padding,tuple) else (model.padding,model.padding)
            d_h,d_w=model.dilation if isinstance(model.dilation,tuple) else (model.dilation,model.dilation)
            ceil_mode=model.ceil_mode
            inner.extend([k_h,k_w,s_h,s_w,p_h,p_w,d_h,d_w,ceil_mode])
        elif op_type=='upsample':
            scale_h,scale_w=model.scale_factor if isinstance(model.scale_factor,tuple) else (model.scale_factor,model.scale_factor)
            mode=model.mode
            inner.extend([scale_h,scale_w,mode])
        config.append(inner)
    return config

config=generate_config(atomic_layers_ordered_completed)

In [ ]:
# conv:                         [op_type, in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias]
# bn:                           [op_type, num_features, eps, momentum, affine, track_running_stats]
# activation:                   [op_type, inplace]
# pool:                         [op_type, k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, ceil_mode]
# upsample:                     [op_type, scale_h, scale_w, mode]
# concat/resadd/c3/sppf/detect: [op_type] (no params)
config

In [ ]:
# make weights list
weights=[]

state_dict = net.state_dict()
weights_list_unordered_incomplete = [{ "name": k, "weight": v.cpu()} for k, v in state_dict.items()]

name_to_weight={w["name"]:w["weight"] for w in weights_list_unordered_incomplete}
for layer in atomic_layers_ordered_completed:
    fname=layer["full_name"]
    inner=[]
    if fname!="x":
        prefix="model."+fname
        for k,v in name_to_weight.items():
            if k.startswith(prefix):
                inner.append(v)
    weights.append(inner)

In [ ]:
weights

for i,layer in enumerate(weights):
    print(f"layer {i}:")
    if not layer:
        print("  []")
    else:
        for j,t in enumerate(layer):
            print(f"  tensor {j} shape={tuple(t.shape)}")

In [ ]:
# creating the DAG

# part I: assign a empty list for every item in atomic_layers_ordered_completed
graph=[[] for _ in atomic_layers_ordered_completed]

In [ ]:
def print_list(lst):
    for i,item in enumerate(lst):print(i,":",item)
print_list(graph)

In [ ]:
# Prepare a copy of atomic_layers with weights
aligned_layers = []

for layer in atomic_layers:
    full_name = layer['full_name']
    matched_weights = []

    # Rule: name contains "model." + full_name + "."
    prefix = f"model.{full_name}."
    for w in weights_list:
        if w['name'].startswith(prefix):
            matched_weights.append(w)  # keep as is

    # Save aligned entry
    layer_entry = layer.copy()
    layer_entry['weights'] = matched_weights  # can be empty list if no match
    aligned_layers.append(layer_entry)
aligned_layers


In [ ]:
def vis_layer(layer):
    print(f"Layer: {layer['full_name']}, name: {type(layer['model']).__name__}, ly: {layer['model']}")
    if layer['weights']:
        for w in layer['weights']:
            print(f"  weight name: {w['name']}, shape: {tuple(w['weight'].shape)}")
    else:
        print("  no weights")
    print("--------------------")

# Visualize first 10 layers
for i in range(30):
    vis_layer(aligned_layers[i])

In [ ]:
len(aligned_layers)

In [ ]:
atomic_graph = {layer['full_name']: [] for layer in atomic_layers}
atomic_graph['0.conv'] = [-1]
atomic_graph

In [ ]:
layer_graph = {}

for i, m in enumerate(net.model):
    if hasattr(m, 'f'):
        # ensure f is a list
        if isinstance(m.f, int):
            from_layers = [m.f]
        else:
            from_layers = m.f if len(m.f) > 0 else [-1]
    else:
        from_layers = [-1]  # default for layers without f
    layer_graph[i] = from_layers

layer_graph

In [ ]:
for i in list(layer_graph.keys()):
    for idx, j in enumerate(layer_graph[i]):
        if j == -1 and layer_graph[i] != 0:
            layer_graph[i][idx] = i+j
layer_graph

In [ ]:
atomic_port = {}

for atomic in atomic_layers:
    full_name = atomic['full_name']
    layer_num = int(full_name.split('.')[0])  # get the first number before the first dot

    if layer_num not in atomic_port:
        # first atomic layer for this layer -> entry
        atomic_port[layer_num] = [full_name, full_name]  # [entry, exit]
    else:
        # update exit as we encounter more atomic layers in this layer
        atomic_port[layer_num][1] = full_name

# convert the list to tuple for (entry, exit)
atomic_port = {k: tuple(v) for k, v in atomic_port.items()}

# print the result
for k in sorted(atomic_port):
    print(k, atomic_port[k])

In [ ]:
for layer, from_layers in layer_graph.items():
    for from_layer in from_layers:
        if from_layer != -1:
            port = atomic_port[layer][0]
            exit_port = atomic_port[from_layer][1]  # 1 = exit
            atomic_graph[port].append(exit_port)
atomic_graph

In [ ]:
keys_in_order = list(atomic_graph.keys())

prev_key = None
for key in keys_in_order:
    # Check if this key is the entry in atomic_port
    is_entry = any(key == entry for entry, _ in atomic_port.values())
    
    if not is_entry and prev_key is not None:
        atomic_graph[key].append(prev_key)
    
    prev_key = key
atomic_graph

In [ ]:
from graphviz import Digraph


dot = Digraph(comment='Atomic Graph')

# Add nodes
for node in atomic_graph.keys():
    dot.node(node, node)

# Add edges
for node, inputs in atomic_graph.items():
    for inp in inputs:
        if inp != -1:  # skip -1 input placeholder
            dot.edge(inp, node)
dot.render(filename='model/atomic_graph', cleanup=True)

In [ ]:
atomic_layers_names = list(atomic_graph.keys())
atomic_graph
# atomic_layers_names

In [ ]:
# Suppose atomic_graph is your dictionary with full names as keys and dependencies
# Example: {'0.conv': [-1], '0.bn': ['0.conv'], ... }

# Map full names to integer indices
name_to_idx = {name: idx for idx, name in enumerate(atomic_layers_names)}

# Convert graph to integer indices
graph = {}

for node, parents in atomic_graph.items():
    # Convert node name to its index
    print(node, name_to_idx[node])
    node_idx = name_to_idx[node]
    
    # Convert parents to indices; handle -1 as a special input node
    parent_indices = []
    for p in parents:
        if isinstance(p, int) and p == -1:
            parent_indices.append(-1)  # keep input node as -1
        else:
            parent_indices.append(name_to_idx[p])
    
    graph[node_idx] = parent_indices

graph

In [ ]:
from graphviz import Digraph

dot = Digraph(comment='Integer Graph')

# Add nodes
for node_idx in graph.keys():
    dot.node(str(node_idx), str(node_idx))  # Graphviz nodes must be strings

# Add edges
for node_idx, parent_indices in graph.items():
    for parent_idx in parent_indices:
        if parent_idx != -1:  # skip input placeholder
            dot.edge(str(parent_idx), str(node_idx))
dot.render(filename='model/graph', cleanup=True)

In [ ]:
aligned_layers_mapped = []
atomic_name_to_idx = {name: idx for idx, name in enumerate(atomic_layers_names)}
for layer in aligned_layers:
    # Map full_name → node integer
    full_name = layer['full_name']
    node_idx = atomic_name_to_idx.get(full_name, -1)
    
    new_layer = layer.copy()
    new_layer['node'] = node_idx
    del new_layer['full_name']
    
    # Keep only last part after last dot
    new_weights = []
    for w in layer['weights']:
        short_name = w['name'].split('.')[-1]  # take last segment
        new_w = w.copy()
        new_w['name'] = short_name
        new_weights.append(new_w)
    
    new_layer['weights'] = new_weights
    aligned_layers_mapped.append(new_layer)
aligned_layers_mapped

In [ ]:
for layer in aligned_layers_mapped:
    print(f"Node: {layer['node']}, Model: {type(layer['model']).__name__}")
    for w in layer['weights']:
        print(f"  Weight name: {w['name']}, shape: {tuple(w['weight'].shape)}")
    print('-'*50)

In [ ]:
operations = [type(layer['model']).__name__ for layer in aligned_layers_mapped]
operations 

In [ ]:
config_dict = {}

for idx, layer in enumerate(aligned_layers_mapped):
    op = layer['model']
    op_type = type(op).__name__
    
    if op_type == "Conv2d":
        # Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias)
        in_ch = op.in_channels
        out_ch = op.out_channels
        k_h, k_w = op.kernel_size if isinstance(op.kernel_size, tuple) else (op.kernel_size, op.kernel_size)
        s_h, s_w = op.stride if isinstance(op.stride, tuple) else (op.stride, op.stride)
        p_h, p_w = op.padding if isinstance(op.padding, tuple) else (op.padding, op.padding)
        bias = op.bias is not None
        config_dict[idx] = [in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias]
        
    elif op_type == "BatchNorm2d":
        # BatchNorm2d(num_features, eps, momentum, affine, track_running_stats)
        num_feat = op.num_features
        eps = op.eps
        momentum = op.momentum
        affine = op.affine
        track_stats = op.track_running_stats
        config_dict[idx] = [num_feat, eps, momentum, affine, track_stats]
        
    elif op_type == "SiLU":
        # SiLU(inplace)
        inplace = op.inplace
        config_dict[idx] = [inplace]
        
    elif op_type == "MaxPool2d":
        # MaxPool2d(kernel_size, stride, padding, dilation, ceil_mode)
        k_h, k_w = op.kernel_size if isinstance(op.kernel_size, tuple) else (op.kernel_size, op.kernel_size)
        s_h, s_w = op.stride if isinstance(op.stride, tuple) else (op.stride, op.stride)
        p_h, p_w = op.padding if isinstance(op.padding, tuple) else (op.padding, op.padding)
        d_h, d_w = op.dilation if isinstance(op.dilation, tuple) else (op.dilation, op.dilation)
        config_dict[idx] = [k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, op.ceil_mode]
        
    elif op_type == "Upsample":
        # Upsample(scale_factor, mode)
        scale_h, scale_w = op.scale_factor if isinstance(op.scale_factor, tuple) else (op.scale_factor, op.scale_factor)
        config_dict[idx] = [scale_h, scale_w, op.mode]
        
    elif op_type == "Concat":
        # Concat usually no parameters
        config_dict[idx] = []
        
    else:
        # Unknown op fallback
        config_dict[idx] = []

config_dict

In [ ]:
config = [config_dict[idx] for idx in range(len(aligned_layers_mapped))]
config

In [ ]:
weights = [
    [w['weight'] for w in layer['weights']]  # list of tensors for this node
    for layer in aligned_layers_mapped
]
weights

In [ ]:
for idx, node_weights in enumerate(weights):
    print(f"Node {idx}:")
    if node_weights:  # if node has weights
        for w_idx, w in enumerate(node_weights):
            print(f"  Weight {w_idx} shape: {tuple(w.shape)}")
    else:
        print("  No weights")
    print("-" * 50)

In [ ]:
# Conv2d: 
# Config = [in_ch, out_ch, k_h, k_w, stride_h, stride_w, pad_h, pad_w, bias_flag]; 
# Weight = [weight(out_ch x in_ch x kH x kW), bias(out_ch) if exists];

# BatchNorm2d: Config = [num_features, eps, momentum, affine_flag, track_running_stats_flag];
# Weight = [weight(num_features), bias(num_features), running_mean(num_features), running_var(num_features)]

# SiLU: Config = [inplace_flag];
# Weight = []

# MaxPool2d: Config = [kernel_h, kernel_w, stride_h, stride_w, pad_h, pad_w, dilation_h, dilation_w, ceil_mode_flag];
# Weight = []

# Upsample: Config = [scale_h, scale_w, mode];
# Weight = []

# Concat: Config = [];
# Weight = []

graph
operations
config
weights


np.save("model/graph.npy", np.array(graph, dtype=object))
np.save("model/operations.npy", np.array(operations, dtype=object))
np.save("model/config.npy", np.array(config, dtype=object))
np.save("model/weights.npy", np.array(weights, dtype=object))